In [ ]:
from PreRun import PreRun
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error as mse
from itertools import product
from datetime import date, datetime
from by_dates_Kfold import k_fold_split_option_a


In [8]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [9]:
params = {
    'objective': 'regression',
    'task': 'train',
    'num_leaves': 10,
    'learning_rate': 0.1,
    'max_depth': 10,
    'num_iterations': 1000,
    'metrics': 'mse',
}

In [10]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [11]:
start_stop_dict = {
    system_id: {
        name[0]: [0, 0] for name in name_val
    } for system_id in systems_good_timezones_manual_edit
}
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        start_stop_dict[system_id].pop(name)
        continue
    if val == 'None':
        real_val = None
    else:
        real_val = val
    prerun_system = PreRun(system_id, f'./test_results/{system_id}-{val}/', real_val, systems_cleaned)
    start_stop_dict[system_id][name][0] = prerun_system.data.at[0, 'time'].date()
    last_index = prerun_system.data.index[-1]
    start_stop_dict[system_id][name][1] = prerun_system.data.at[last_index, 'time'].date()

In [12]:
start_stop_dict

{4: {'other': [datetime.date(2007, 9, 1), datetime.date(2023, 2, 28)]},
 10: {'other': [datetime.date(2006, 1, 25), datetime.date(2023, 2, 28)]},
 33: {'other': [datetime.date(2010, 11, 10), datetime.date(2023, 2, 28)]},
 36: {'other': [datetime.date(2012, 3, 30), datetime.date(2019, 7, 21)]},
 50: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 51: {'other': [datetime.date(1994, 5, 14), datetime.date(2023, 2, 28)]},
 1199: {'inverter': [datetime.date(2010, 5, 29), datetime.date(2018, 8, 3)]},
 1204: {'inverter': [datetime.date(2011, 2, 9), datetime.date(2015, 1, 5)]},
 1283: {'inverter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)],
  'meter': [datetime.date(2012, 1, 10), datetime.date(2022, 2, 8)]},
 1284: {'other': [datetime.date(2012, 6, 30), datetime.date(2015, 3, 15)]},
 1289: {'other': [datetime.date(2012, 9, 28), datetime.date(2020, 5, 13)]},
 1332: {'inverter': [datetime.date(2013, 3, 30), datetime.date(2014, 7, 31)],
  'meter': [datetime.date(

In [13]:
def basic_lightbgm_test(system_id: int, read_path: str, met_or_inv, systems_cleaned: pd.DataFrame, params: dict):
    prerun_system = PreRun(system_id, read_path, met_or_inv, systems_cleaned)
    prerun_system.add_energy_features_only(daily_lags=1, include_last_year=True, todays_lags=1, include_hour_cyclic=True,include_day_of_year_cyclic=True)
    prerun_system.add_weather_features_only()
    df = prerun_system.amended_data
    df['year'] = df['time'].dt.year
    my_cols = ['year', 'hour_sin', 'hour_cos', 'day_of_year_sin', 'day_of_year_cos', 'last_year', '1_days_ago', '1_hours_ago_today', 'cloud_cover', 'global_tilted_irradiance', 'proportion_daytime']
    df_starter_train = df[df['time'] < pd.Timestamp(year=2018, month=1, day=1, hour=0)]
    X_starter_train = df_starter_train[my_cols]
    y_starter_train = df_starter_train['energy']
    lgb_tr = lgb.Dataset(X_starter_train, label=y_starter_train)
    df_starter_val = df[(df['time'] >= pd.Timestamp(year=2019, month=1, day=1, hour=0))
                        & (df['time'] < pd.Timestamp(year=2021, month=1, day=1, hour=0))]
    X_val = df_starter_val[my_cols]
    y_val = df_starter_val['energy']
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_tr)
    df_starter_test = df[df['time'] >= pd.Timestamp(year=2022, month=1, day=1, hour=0)]
    X_test = df_starter_test[my_cols]
    y_test = df_starter_test['energy']
    num_round = 100
    bst = lgb.train(params, lgb_tr, num_round, valid_sets=[lgb_val,])
    y_pred = bst.predict(X_test)
    print(f'Mean squared error: {mse(y_test, y_pred):.5f}')
    

In [14]:
basic_lightbgm_test(4, './test_results/4-None', None, systems_cleaned, params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001072 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1923
[LightGBM] [Info] Number of data points in the train set: 27796, number of used features: 11
[LightGBM] [Info] Start training from score 0.393301
Mean squared error: 0.00957


In [15]:
systems_good_timezones_manual_edit = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]

In [16]:
name_val = (('inverter', 'inverter'), ('meter', 'meter'), ('other', 'None'))

In [17]:
from itertools import product

In [12]:
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        continue
    print(f'{system_id}-{name}')
    if val == 'None':
        real_val = None
    else:
        real_val = val
    print(basic_lightbgm_test(system_id, f'./test_results/{system_id}-{val}', real_val, systems_cleaned, params))

4-other
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000284 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1923
[LightGBM] [Info] Number of data points in the train set: 27796, number of used features: 11
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Info] Start training from score 0.393301
Mean squared error: 0.00957
None
10-other
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignored. Current value: learning_rate=0.1
[LightGBM] [Warning] learning_rate is set=0.1, shrinkage_rate=0.1 will be ignore

ValueError: Input data must be 2 dimensional and non empty.

In [18]:
for system_id, (name, val) in product(systems_good_timezones_manual_edit, name_val):
    try:
        data_pq = pq.ParquetDataset(
            f'./test_results/{system_id}-{val}/{name}/{system_id}'
        )
        good_days_df = pd.read_csv(
            f'./test_results/{system_id}-{val}/good_days/{system_id}_good_days_{name}.csv'
        )
    except FileNotFoundError:
        continue
    print(f'{system_id}-{name}')
    if val == 'None':
        real_val = None
    else:
        real_val = val
    trial_data = PreRun(system_id, f'./test_results/{system_id}-{val}', real_val, systems_cleaned)
    trial_data.good_end_days_naive(streak=7)
    num_good_days = len(trial_data.good_days)
    num_good_ends = len(trial_data.end_days_naive)
    the_pct = num_good_ends / num_good_days * 100
    print(f'Good ends count: {num_good_ends}')
    print(f'Good days count: {num_good_days}')
    print(f'Ratio: {the_pct:.5f}%')
    print('')


4-other
Good ends count: 1127
Good days count: 3598
Ratio: 31.32296%

10-other
Good ends count: 891
Good days count: 4170
Ratio: 21.36691%

33-other
Good ends count: 1484
Good days count: 3594
Ratio: 41.29104%

36-other
Good ends count: 358
Good days count: 1510
Ratio: 23.70861%

50-other
Good ends count: 2062
Good days count: 5817
Ratio: 35.44783%

51-other
Good ends count: 1784
Good days count: 5644
Ratio: 31.60879%

1199-inverter
Good ends count: 828
Good days count: 2275
Ratio: 36.39560%

1204-inverter
Good ends count: 428
Good days count: 1164
Ratio: 36.76976%

1283-inverter
Good ends count: 857
Good days count: 2124
Ratio: 40.34840%

1283-meter
Good ends count: 864
Good days count: 2136
Ratio: 40.44944%

1284-other
Good ends count: 177
Good days count: 746
Ratio: 23.72654%

1289-other
Good ends count: 322
Good days count: 1774
Ratio: 18.15107%

1332-inverter
Good ends count: 110
Good days count: 310
Ratio: 35.48387%

1332-meter
Good ends count: 937
Good days count: 2185
Ratio: 42

In [19]:
shorter_test = PreRun(4903, f'./test_results/4903-inverter', 'inverter', systems_cleaned)
shorter_test.add_weather_features_only()

In [20]:
shorter_test.good_days.iloc[30:40]

,date
30,2014-09-23
31,2014-09-24
32,2014-09-26
33,2014-09-27
34,2014-09-28
35,2014-09-29
36,2014-09-30
37,2014-10-01
38,2014-10-02
39,2014-10-03


In [21]:
shorter_test.good_end_days_naive(streak=7)

,date
0,2014-09-11
1,2014-09-12
2,2014-09-13
3,2014-09-14
4,2014-09-15
...,...
422,2017-12-06
423,2017-12-07
424,2017-12-08
425,2017-12-22


In [22]:
good_days_ext = shorter_test.good_days
end_days_ext = shorter_test.end_days_naive

In [23]:
end_days_ext['date_again'] = end_days_ext['date'].dt.date

In [24]:
end_days_ext.at[0, 'date_again']

datetime.date(2014, 9, 11)

In [25]:
end_days_ext_b = end_days_ext[['date_again']]
end_days_ext_b = end_days_ext_b.rename(columns={'date_again': 'date'})

In [26]:
type(end_days_ext.at[0, 'date'])

pandas.Timestamp

In [27]:
shorter_test.add_energy_features_only(daily_lags=2, include_hour_cyclic=True, include_day_of_year_cyclic=True)

,time,energy,cloud_cover,global_tilted_irradiance,proportion_daytime,day,1_days_ago,2_days_ago,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos
0,2014-08-02 00:00:00,6.500000,1.00,0.000000,0.0,2014-08-02,NaN,NaN,-0.501242,-0.865307,0.000000e+00,1.000000
1,2014-08-02 01:00:00,6.500000,1.00,0.000000,0.0,2014-08-02,NaN,NaN,-0.501242,-0.865307,2.588190e-01,0.965926
2,2014-08-02 02:00:00,6.500000,0.99,0.000000,0.0,2014-08-02,NaN,NaN,-0.501242,-0.865307,5.000000e-01,0.866025
3,2014-08-02 03:00:00,6.500000,0.90,0.000000,0.0,2014-08-02,NaN,NaN,-0.501242,-0.865307,7.071068e-01,0.707107
4,2014-08-02 04:00:00,6.500000,0.88,0.000000,0.0,2014-08-02,NaN,NaN,-0.501242,-0.865307,8.660254e-01,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...
12474,2018-02-16 12:00:00,7.744983,1.00,71.207573,1.0,2018-02-16,24.376500,30.737833,0.711657,0.702527,1.224647e-16,-1.000000
12475,2018-02-16 13:00:00,6.969800,0.99,110.173233,1.0,2018-02-16,33.824833,40.110167,0.711657,0.702527,-2.588190e-01,-0.965926
12476,2018-02-16 14:00:00,3.391600,1.00,66.241890,1.0,2018-02-16,15.954633,22.937667,0.711657,0.702527,-5.000000e-01,-0.866025
12477,2018-02-16 15:00:00,3.359783,0.97,56.880394,1.0,2018-02-16,10.527367,13.005900,0.711657,0.702527,-7.071068e-01,-0.707107


In [28]:
am_dat_ext = shorter_test.amended_data

In [29]:
am_dat_ext.loc[:, 'year'] = am_dat_ext['time'].dt.year

In [30]:
am_dat_ext.head()

,time,energy,cloud_cover,global_tilted_irradiance,proportion_daytime,1_days_ago,2_days_ago,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,year
0,2014-08-02 00:00:00,6.5,1.00,0.0,0.0,NaN,NaN,-0.501242,-0.865307,0.000000,1.000000,2014
1,2014-08-02 01:00:00,6.5,1.00,0.0,0.0,NaN,NaN,-0.501242,-0.865307,0.258819,0.965926,2014
2,2014-08-02 02:00:00,6.5,0.99,0.0,0.0,NaN,NaN,-0.501242,-0.865307,0.500000,0.866025,2014
3,2014-08-02 03:00:00,6.5,0.90,0.0,0.0,NaN,NaN,-0.501242,-0.865307,0.707107,0.707107,2014
4,2014-08-02 04:00:00,6.5,0.88,0.0,0.0,NaN,NaN,-0.501242,-0.865307,0.866025,0.500000,2014


In [31]:
am_dat_ext['energy'].max()

np.float64(62.87533333333334)

In [32]:
df_train = am_dat_ext[am_dat_ext['time'] < pd.Timestamp(datetime(2017, 1, 1))]
df_test = am_dat_ext[am_dat_ext['time'] >= pd.Timestamp(datetime(2017, 1, 1))]

In [33]:
my_splits = k_fold_split_option_a(
    df_train, end_days_ext, 50, None, 'front', 'index'
)

In [34]:
my_cols = ['year', 'hour_cos', 'hour_sin', 'day_of_year_cos', 'day_of_year_sin', 'proportion_daytime', 'global_tilted_irradiance', 'cloud_cover']

In [35]:
X_train = df_train[my_cols]
y_train = df_train['energy']

In [36]:
X_test = df_test[my_cols]
y_test = df_test['energy']

In [37]:
my_splits

[(RangeIndex(start=0, stop=477, step=1),
  RangeIndex(start=489, stop=501, step=1)),
 (RangeIndex(start=0, stop=561, step=1),
  RangeIndex(start=573, stop=584, step=1)),
 (RangeIndex(start=0, stop=573, step=1),
  RangeIndex(start=584, stop=596, step=1)),
 (RangeIndex(start=0, stop=584, step=1),
  RangeIndex(start=596, stop=608, step=1)),
 (RangeIndex(start=0, stop=596, step=1),
  RangeIndex(start=608, stop=620, step=1)),
 (RangeIndex(start=0, stop=608, step=1),
  RangeIndex(start=620, stop=632, step=1)),
 (RangeIndex(start=0, stop=620, step=1),
  RangeIndex(start=632, stop=643, step=1)),
 (RangeIndex(start=0, stop=632, step=1),
  RangeIndex(start=643, stop=655, step=1)),
 (RangeIndex(start=0, stop=643, step=1),
  RangeIndex(start=655, stop=667, step=1)),
 (RangeIndex(start=0, stop=655, step=1),
  RangeIndex(start=667, stop=678, step=1)),
 (RangeIndex(start=0, stop=667, step=1),
  RangeIndex(start=678, stop=689, step=1)),
 (RangeIndex(start=0, stop=678, step=1),
  RangeIndex(start=689, 

In [38]:
end_days_ext_b.iloc[200:210]

,date
200,2016-03-03
201,2016-03-11
202,2016-03-12
203,2016-03-26
204,2016-03-27
205,2016-03-28
206,2016-03-29
207,2016-03-30
208,2016-03-31
209,2016-04-19


In [39]:
from sklearn.linear_model import RidgeCV

In [40]:
test_cv = RidgeCV(alphas=(0.001, 0.01, 0.01, 1, 10, 100, 1000),
                  fit_intercept=True, scoring='neg_mean_squared_error',
                  cv=my_splits)

In [41]:
test_cv.fit(X_train, y_train)

,"alphas alphas: array-like of shape (n_alphas,), default=(0.1, 1.0, 10.0)Array of alpha values to try.Regularization strength; must be a positive float. Regularizationimproves the conditioning of the problem and reduces the variance ofthe estimates. Larger values specify stronger regularization.Alpha corresponds to ``1 / (2C)`` in other linear models such as:class:`~sklearn.linear_model.LogisticRegression` or:class:`~sklearn.svm.LinearSVC`.If using Leave-One-Out cross-validation, alphas must be strictly positive.","(0.001, ...)"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"scoring scoring: str, callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: negative :ref:`mean squared error ` if cv is None (i.e. when using leave-one-out cross-validation), or :ref:`coefficient of determination ` (:math:`R^2`) otherwise.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the efficient Leave-One-Out cross-validation- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used, else,:class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.","[(RangeIndex(st...p=477, step=1), ...), (RangeIndex(st...p=561, step=1), ...), ...]"
,"gcv_mode gcv_mode: {'auto', 'svd', 'eigen'}, default='auto'Flag indicating which strategy to use when performingLeave-One-Out Cross-Validation. Options are:: 'auto' : use 'svd' if n_samples > n_features, otherwise use 'eigen' 'svd' : force use of singular value decomposition of X when X is dense, eigenvalue decomposition of X^T.X when X is sparse. 'eigen' : force computation via eigendecomposition of X.X^TThe 'auto' mode is the default and is intended to pick the cheaperoption of the two depending on the shape of the training data.",None
,"store_cv_results store_cv_results: bool, default=FalseFlag indicating if the cross-validation values corresponding toeach alpha should be stored in the ``cv_results_`` attribute (seebelow). This flag is only compatible with ``cv=None`` (i.e. usingLeave-One-Out Cross-Validation)... versionchanged:: 1.5 Parameter name changed from `store_cv_values` to `store_cv_results`.",False
,"alpha_per_target alpha_per_target: bool, default=FalseFlag indicating whether to optimize the alpha value (picked from the`alphas` parameter list) for each target separately (for multi-outputsettings: multiple prediction targets). When set to `True`, afterfitting, the `alpha_` attribute will contain a value for each target.When set to `False`, a single alpha is used for all targets... versionadded:: 0.24",False


In [42]:
y_pred = test_cv.predict(X_test)

In [43]:
from sklearn.metrics import root_mean_squared_error as rmse

In [44]:
rmse(y_true=y_test, y_pred=y_pred)

7.83775732654247

In [45]:
for train_split, test_split in my_splits:
    X_tt = X_train.loc[train_split]
    y_tt = y_train.loc[train_split]
    X_ho = X_train.loc[test_split]
    y_ho = y_train.loc[test_split]
    

In [46]:
my_splits_alt = k_fold_split_option_a(
    df_train, end_days_ext, 50, 5, 'back', 'DataFrame'
)

In [47]:
my_splits_alt[12][1].tail()

,time,energy,cloud_cover,global_tilted_irradiance,proportion_daytime,1_days_ago,2_days_ago,day_of_year_sin,day_of_year_cos,hour_sin,hour_cos,year,date
7899,2016-09-23 13:00:00,40.304333,0.0,843.964172,1.0,41.548833,31.008500,-0.989372,-0.145404,-0.258819,-0.965926,2016,2016-09-23
7900,2016-09-23 14:00:00,33.709833,0.0,777.991699,1.0,34.767167,27.843333,-0.989372,-0.145404,-0.500000,-0.866025,2016,2016-09-23
7901,2016-09-23 15:00:00,24.946333,0.0,646.937073,1.0,24.927333,14.831833,-0.989372,-0.145404,-0.707107,-0.707107,2016,2016-09-23
7902,2016-09-23 16:00:00,11.707750,0.0,466.629364,1.0,12.205267,7.335167,-0.989372,-0.145404,-0.866025,-0.500000,2016,2016-09-23
7903,2016-09-23 17:00:00,0.636600,0.0,257.975922,1.0,0.443133,1.757967,-0.989372,-0.145404,-0.965926,-0.258819,2016,2016-09-23


In [48]:
train_split

RangeIndex(start=0, stop=2063, step=1)

In [49]:
test_split

RangeIndex(start=2076, stop=2089, step=1)

In [50]:
rng = np.random.default_rng()

In [51]:
x = np.linspace(0, 10000, 101).T

In [52]:
y = 20*x + 100 + rng.normal(loc=0,scale=1,size=(101,))

In [53]:
x_ser = pd.Series(x, name='x')
y_ser = pd.Series(y, name='y')
merger = pd.merge(x_ser, y_ser, left_index=True, right_index=True)

In [54]:
merger.head()

,x,y
0,0.0,100.728597
1,100.0,2099.853529
2,200.0,4099.198991
3,300.0,6099.076381
4,400.0,8099.192700


In [55]:
from sklearn.model_selection import KFold

In [56]:
kfold = KFold(n_splits=5)
my_k_fold_splits = kfold.split(merger[['x']], merger['y'])

In [57]:
for train_idx, test_idx in my_k_fold_splits:
    print(type(train_idx))
    print(train_idx.shape)
    print(f'Training indices: {train_idx}')
    print(f'Testing indices: {test_idx}')

<class 'numpy.ndarray'>
(80,)
Training indices: [ 21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73  74
  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92
  93  94  95  96  97  98  99 100]
Testing indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
<class 'numpy.ndarray'>
(81,)
Training indices: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55
  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73
  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91
  92  93  94  95  96  97  98  99 100]
Testing indices: [21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40]
<class 'numpy.ndarray'>
(81,)
Training indices: [  0   1   2   3   4   5   6   7   8   

In [58]:
my_test_splits

NameError: name 'my_test_splits' is not defined

In [ ]:
my_train_0, my_test_0 = my_test_splits[0]

In [ ]:
my_train_0

'x'

In [ ]:
type(my_train_0)

str

In [ ]:
x

array([    0.,   100.,   200.,   300.,   400.,   500.,   600.,   700.,
         800.,   900.,  1000.,  1100.,  1200.,  1300.,  1400.,  1500.,
        1600.,  1700.,  1800.,  1900.,  2000.,  2100.,  2200.,  2300.,
        2400.,  2500.,  2600.,  2700.,  2800.,  2900.,  3000.,  3100.,
        3200.,  3300.,  3400.,  3500.,  3600.,  3700.,  3800.,  3900.,
        4000.,  4100.,  4200.,  4300.,  4400.,  4500.,  4600.,  4700.,
        4800.,  4900.,  5000.,  5100.,  5200.,  5300.,  5400.,  5500.,
        5600.,  5700.,  5800.,  5900.,  6000.,  6100.,  6200.,  6300.,
        6400.,  6500.,  6600.,  6700.,  6800.,  6900.,  7000.,  7100.,
        7200.,  7300.,  7400.,  7500.,  7600.,  7700.,  7800.,  7900.,
        8000.,  8100.,  8200.,  8300.,  8400.,  8500.,  8600.,  8700.,
        8800.,  8900.,  9000.,  9100.,  9200.,  9300.,  9400.,  9500.,
        9600.,  9700.,  9800.,  9900., 10000.])

In [ ]:
X_ho

,year,hour_cos,hour_sin,day_of_year_cos,day_of_year_sin,proportion_daytime,global_tilted_irradiance,cloud_cover
2076,2015,6.123234e-17,1.000000e+00,-0.047321,0.99888,1.000000,0.000000,0.09
2077,2015,-2.588190e-01,9.659258e-01,-0.047321,0.99888,1.000000,87.156464,0.39
2078,2015,-5.000000e-01,8.660254e-01,-0.047321,0.99888,1.000000,291.312225,0.82
2079,2015,-7.071068e-01,7.071068e-01,-0.047321,0.99888,1.000000,469.264862,0.65
2080,2015,-8.660254e-01,5.000000e-01,-0.047321,0.99888,1.000000,622.256775,0.88
2081,2015,-9.659258e-01,2.588190e-01,-0.047321,0.99888,1.000000,753.680115,0.97
2082,2015,-1.000000e+00,1.224647e-16,-0.047321,0.99888,1.000000,824.020569,0.84
2083,2015,-9.659258e-01,-2.588190e-01,-0.047321,0.99888,1.000000,854.397278,0.39
2084,2015,-8.660254e-01,-5.000000e-01,-0.047321,0.99888,1.000000,903.676331,0.02
2085,2015,-7.071068e-01,-7.071068e-01,-0.047321,0.99888,1.000000,770.826172,0.05


In [ ]:
end_days_ext['diff'] = end_days_ext['date'].diff(periods=1)

In [ ]:
good_days_ext['diff'] = good_days_ext['date'].diff(periods=1)

In [ ]:
good_days_ext.iloc[32:52]

,date,diff
32,2014-09-26,2 days
33,2014-09-27,1 days
34,2014-09-28,1 days
35,2014-09-29,1 days
36,2014-09-30,1 days
37,2014-10-01,1 days
38,2014-10-02,1 days
39,2014-10-03,1 days
40,2014-10-04,1 days
41,2014-10-05,1 days


In [ ]:
end_days_ext.iloc[5:21]

,date,date_again,diff
5,2014-09-16,2014-09-16,1 days
6,2014-09-24,2014-09-24,8 days
7,2014-10-02,2014-10-02,8 days
8,2014-10-03,2014-10-03,1 days
9,2014-10-04,2014-10-04,1 days
10,2014-10-05,2014-10-05,1 days
11,2014-10-06,2014-10-06,1 days
12,2014-10-07,2014-10-07,1 days
13,2014-10-08,2014-10-08,1 days
14,2014-10-09,2014-10-09,1 days


In [ ]:
shorter_test.end_days_naive.diff(periods=1).iloc[10:20]

,date,date_again,diff
10,1 days,"1 day, 0:00:00",0 days
11,1 days,"1 day, 0:00:00",0 days
12,1 days,"1 day, 0:00:00",0 days
13,1 days,"1 day, 0:00:00",0 days
14,1 days,"1 day, 0:00:00",0 days
15,1 days,"1 day, 0:00:00",0 days
16,1 days,"1 day, 0:00:00",0 days
17,1 days,"1 day, 0:00:00",0 days
18,1 days,"1 day, 0:00:00",0 days
19,1 days,"1 day, 0:00:00",0 days
